# Day 12: Project — Complete Sentiment Classifier

**Goal:** Bring everything from Days 4, 8, 9, 10, 11 into a single end-to-end project.

### What we'll do

Build a sentiment classifier that:
1. Loads text data
2. Tokenizes & builds vocabulary
3. Trains 3 different models and compares them
4. Saves the best one
5. Provides a `predict()` function for new text
6. Performs error analysis on failures

### The 3 models we'll compare

| Model | Architecture |
|-------|-------------|
| **A: BoW + Linear** | One-hot counts → 1 Linear layer (Day 10 baseline) |
| **B: Embedding + Mean** | Embedding lookup → mean pool → 1 Linear (Day 11) |
| **C: Embedding + MLP** | Embedding lookup → mean pool → 2 hidden layers (deeper) |

This "always compare against a baseline" habit is one of the most important things in ML.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import json
import os
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## Step 1: A Larger Sentiment Dataset

200 reviews — bigger than Day 10/11 (30 reviews) but still small enough to inspect.

In [ ]:
# Generate a larger sentiment dataset by combining base reviews with templates

POSITIVE_TEMPLATES = [
    "This {noun} was {pos_adj}! {pos_phrase}",
    "{pos_adj} {noun}. I {pos_verb} every minute.",
    "What a {pos_adj} {noun}. {pos_phrase}",
    "I {pos_verb} this {noun}. {pos_phrase}",
    "{pos_phrase} A {pos_adj} {noun}.",
    "Truly {pos_adj}. {pos_phrase}",
    "The {noun} was {pos_adj}. The {component} was {pos_adj} too.",
    "I would recommend this {noun}. It's {pos_adj}.",
    "{pos_adj} from start to finish. {pos_phrase}",
    "A {pos_adj} and engaging {noun}.",
]

NEGATIVE_TEMPLATES = [
    "This {noun} was {neg_adj}. {neg_phrase}",
    "{neg_adj} {noun}. I {neg_verb} every minute.",
    "What a {neg_adj} {noun}. {neg_phrase}",
    "I {neg_verb} this {noun}. {neg_phrase}",
    "{neg_phrase} A {neg_adj} {noun}.",
    "Truly {neg_adj}. {neg_phrase}",
    "The {noun} was {neg_adj}. The {component} was {neg_adj} too.",
    "Do not watch this {noun}. It's {neg_adj}.",
    "{neg_adj} from start to finish. {neg_phrase}",
    "A {neg_adj} and disappointing {noun}.",
]

POS_ADJ = ["amazing", "wonderful", "fantastic", "brilliant", "beautiful",
           "great", "excellent", "outstanding", "perfect", "delightful",
           "superb", "stunning", "incredible", "phenomenal", "captivating"]
NEG_ADJ = ["terrible", "awful", "boring", "horrible", "dreadful",
           "bad", "poor", "disappointing", "painful", "weak",
           "dull", "stupid", "tedious", "lame", "confusing"]
POS_VERB = ["loved", "enjoyed", "adored"]
NEG_VERB = ["hated", "disliked", "regretted"]
NOUNS = ["movie", "film", "show", "story", "performance"]
COMPONENTS = ["acting", "plot", "writing", "direction", "cast", "score"]
POS_PHRASES = [
    "Highly recommended.", "Would watch again.", "Best of the year.",
    "Beautifully done.", "A real masterpiece.", "Truly inspiring.",
]
NEG_PHRASES = [
    "Skip it.", "Not worth your time.", "Worst of the year.",
    "Painful to sit through.", "A complete disaster.", "Avoid at all costs.",
]

def fill_template(template, label):
    """Fill a template with random words for the given sentiment."""
    return template.format(
        noun=np.random.choice(NOUNS),
        component=np.random.choice(COMPONENTS),
        pos_adj=np.random.choice(POS_ADJ),
        neg_adj=np.random.choice(NEG_ADJ),
        pos_verb=np.random.choice(POS_VERB),
        neg_verb=np.random.choice(NEG_VERB),
        pos_phrase=np.random.choice(POS_PHRASES),
        neg_phrase=np.random.choice(NEG_PHRASES),
    )

# Generate 100 positive + 100 negative reviews
np.random.seed(42)
reviews = []
for _ in range(100):
    template = np.random.choice(POSITIVE_TEMPLATES)
    reviews.append((fill_template(template, 1), 1))
for _ in range(100):
    template = np.random.choice(NEGATIVE_TEMPLATES)
    reviews.append((fill_template(template, 0), 0))

print(f"Total reviews: {len(reviews)}")
print(f"Positive: {sum(1 for _, l in reviews if l == 1)}")
print(f"Negative: {sum(1 for _, l in reviews if l == 0)}")
print(f"\n--- Sample positives ---")
for r, l in reviews[:3]:
    print(f"  '{r}'")
print(f"\n--- Sample negatives ---")
for r, l in reviews[100:103]:
    print(f"  '{r}'")

## Step 2: Tokenizer & Vocabulary

Build a clean Tokenizer class — easier to save, load, and reuse than a bunch of loose functions.

In [ ]:
class Tokenizer:
    """Tokenize text into IDs. Includes <unk> and <pad>."""
    
    UNK = '<unk>'
    PAD = '<pad>'
    
    def __init__(self):
        self.word_to_idx = {}
        self.idx_to_word = {}
        self.vocab_size = 0
    
    def fit(self, texts, min_count=1):
        """Build vocab from a list of texts. Words seen < min_count are <unk>."""
        counts = Counter()
        for t in texts:
            counts.update(self.tokenize(t))
        # Specials first, so <unk>=0, <pad>=1
        self.word_to_idx = {self.UNK: 0, self.PAD: 1}
        for word, c in counts.most_common():
            if c >= min_count:
                self.word_to_idx[word] = len(self.word_to_idx)
        self.idx_to_word = {i: w for w, i in self.word_to_idx.items()}
        self.vocab_size = len(self.word_to_idx)
        return self
    
    @staticmethod
    def tokenize(text):
        text = text.lower()
        text = re.sub(r"[^a-z0-9' ]+", " ", text)
        return text.split()
    
    def encode(self, text, max_len=None):
        """Text → list of token IDs, padded/truncated to max_len if given."""
        unk = self.word_to_idx[self.UNK]
        pad = self.word_to_idx[self.PAD]
        ids = [self.word_to_idx.get(t, unk) for t in self.tokenize(text)]
        if max_len is not None:
            ids = ids[:max_len] + [pad] * max(0, max_len - len(ids))
        return ids
    
    def decode(self, ids):
        return ' '.join([self.idx_to_word[i] for i in ids if i not in {0, 1}])
    
    def save(self, path):
        with open(path, 'w') as f:
            json.dump(self.word_to_idx, f)
    
    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, 'r') as f:
            tok.word_to_idx = json.load(f)
        tok.idx_to_word = {i: w for w, i in tok.word_to_idx.items()}
        tok.vocab_size = len(tok.word_to_idx)
        return tok

# Build tokenizer
texts = [t for t, _ in reviews]
tokenizer = Tokenizer().fit(texts)

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"\nDemo: encode/decode round-trip:")
sample = "What a fantastic film! I loved it."
ids = tokenizer.encode(sample, max_len=15)
decoded = tokenizer.decode(ids)
print(f"  Original: '{sample}'")
print(f"  Token IDs: {ids}")
print(f"  Decoded:   '{decoded}'")

## Step 3: Dataset & DataLoader

Wrap our reviews in PyTorch's `Dataset` so we can batch them with `DataLoader`.

In [ ]:
MAX_LEN = 20

class SentimentDataset(Dataset):
    """Wraps (text, label) pairs and returns (token_ids, label) tensors."""
    
    def __init__(self, reviews, tokenizer, max_len=MAX_LEN):
        self.reviews = reviews
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, idx):
        text, label = self.reviews[idx]
        ids = self.tokenizer.encode(text, max_len=self.max_len)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# 70/15/15 split
np.random.seed(0)
shuffled = np.random.permutation(len(reviews))
n = len(reviews)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_reviews = [reviews[i] for i in shuffled[:train_end]]
val_reviews = [reviews[i] for i in shuffled[train_end:val_end]]
test_reviews = [reviews[i] for i in shuffled[val_end:]]

# Build datasets
train_ds = SentimentDataset(train_reviews, tokenizer)
val_ds = SentimentDataset(val_reviews, tokenizer)
test_ds = SentimentDataset(test_reviews, tokenizer)

# DataLoaders
BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")
print(f"Test:  {len(test_ds)} samples, {len(test_loader)} batches")

# Show one batch
xb, yb = next(iter(train_loader))
print(f"\nFirst batch:")
print(f"  Token IDs shape: {xb.shape}  ({BATCH_SIZE} samples × {MAX_LEN} tokens)")
print(f"  Labels shape:    {yb.shape}")
print(f"  First sample:    '{tokenizer.decode(xb[0].tolist())}' → label={yb[0].item()}")

## Step 4: Define the 3 Models

Each takes `(batch, seq_len)` token IDs and outputs `(batch, 2)` logits.

In [ ]:
PAD_IDX = tokenizer.word_to_idx[Tokenizer.PAD]

# MODEL A — Bag of Words + Linear (Day 10 baseline)
class BoWModel(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, num_classes)
        self.vocab_size = vocab_size
    
    def forward(self, ids):
        # ids: (batch, seq_len) → BoW count vector: (batch, vocab_size)
        bow = F.one_hot(ids, num_classes=self.vocab_size).float().sum(dim=1)
        return self.fc(bow)

# MODEL B — Embedding + Mean Pool (Day 11)
class EmbedMeanModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_classes=2, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, ids):
        # ids: (batch, seq_len) → (batch, seq_len, embed_dim) → (batch, embed_dim)
        embedded = self.embedding(ids)
        sentence_vec = embedded.mean(dim=1)
        return self.fc(sentence_vec)

# MODEL C — Embedding + MLP (deeper)
class EmbedMLPModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64,
                 num_classes=2, pad_idx=PAD_IDX, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, ids):
        embedded = self.embedding(ids)
        sentence_vec = embedded.mean(dim=1)
        x = F.relu(self.fc1(sentence_vec))
        x = self.dropout(x)
        return self.fc2(x)

# Show parameter counts
for name, ModelClass in [('BoW', BoWModel),
                         ('EmbedMean', EmbedMeanModel),
                         ('EmbedMLP', EmbedMLPModel)]:
    m = ModelClass(tokenizer.vocab_size)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:>12}: {n_params:,} parameters")

## Step 5: Reusable Training Function

One function trains any model. Returns the loss/accuracy history + the best-model state dict.

In [ ]:
def evaluate(model, loader, loss_fn):
    """Compute average loss and accuracy on a DataLoader."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for ids, labels in loader:
            logits = model(ids)
            loss = loss_fn(logits, labels)
            total_loss += loss.item() * len(ids)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += len(ids)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=30, lr=0.01, patience=8):
    """Train model with early stopping. Returns history + best state dict."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        # Train step
        model.train()
        epoch_loss = 0
        epoch_n = 0
        for ids, labels in train_loader:
            logits = model(ids)
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(ids)
            epoch_n += len(ids)
        train_loss = epoch_loss / epoch_n
        
        # Val step
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Early stopping + best-model tracking
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    return history, best_state, len(history['train_loss'])

print("Training function defined. Now training all 3 models...")

In [ ]:
# Train all 3 models

print("Training Model A — BoW + Linear (baseline)...")
torch.manual_seed(42)
bow_model = BoWModel(tokenizer.vocab_size)
bow_history, bow_best, bow_epochs = train_model(bow_model, train_loader, val_loader)
bow_model.load_state_dict(bow_best)
print(f"  Done in {bow_epochs} epochs")

print("\nTraining Model B — Embedding + Mean (Day 11)...")
torch.manual_seed(42)
emb_model = EmbedMeanModel(tokenizer.vocab_size, embed_dim=32)
emb_history, emb_best, emb_epochs = train_model(emb_model, train_loader, val_loader)
emb_model.load_state_dict(emb_best)
print(f"  Done in {emb_epochs} epochs")

print("\nTraining Model C — Embedding + MLP (deeper)...")
torch.manual_seed(42)
mlp_model = EmbedMLPModel(tokenizer.vocab_size, embed_dim=32, hidden_dim=64)
mlp_history, mlp_best, mlp_epochs = train_model(mlp_model, train_loader, val_loader)
mlp_model.load_state_dict(mlp_best)
print(f"  Done in {mlp_epochs} epochs")

print("\nAll models trained!")

## Step 6: Compare Models — Test Set Performance

In [ ]:
# Evaluate all 3 models on the test set
loss_fn = nn.CrossEntropyLoss()

results = []
for name, model in [('BoW', bow_model), ('EmbedMean', emb_model), ('EmbedMLP', mlp_model)]:
    test_loss, test_acc = evaluate(model, test_loader, loss_fn)
    n_params = sum(p.numel() for p in model.parameters())
    results.append((name, test_loss, test_acc, n_params))

# Print results table
print(f"{'Model':>12} | {'Params':>9} | {'Test Loss':>9} | {'Test Acc':>8}")
print("-" * 55)
for name, loss, acc, params in results:
    print(f"{name:>12} | {params:>9,} | {loss:>9.4f} | {acc*100:>7.1f}%")

# Plot loss curves & test accuracies
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for hist, name, color in [(bow_history, 'BoW', 'blue'),
                            (emb_history, 'EmbedMean', 'green'),
                            (mlp_history, 'EmbedMLP', 'red')]:
    axes[0].plot(hist['val_loss'], color=color, label=name, linewidth=2)
    axes[1].plot([a*100 for a in hist['val_acc']], color=color, label=name, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val loss')
axes[0].set_title('Validation loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val accuracy (%)')
axes[1].set_title('Validation accuracy'); axes[1].legend()
axes[1].set_ylim(40, 105); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find best model
best_name = max(results, key=lambda r: r[2])[0]
print(f"\n🏆 Best model: {best_name}")

## Step 7: Save Everything for Production

Real-world deployment needs the **model + tokenizer + config**. Save all three:

In [ ]:
ARTIFACTS_DIR = '/tmp/sentiment_model'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Save the best model (EmbedMLP usually wins, but use whichever did best)
best_models = {'BoW': bow_model, 'EmbedMean': emb_model, 'EmbedMLP': mlp_model}
best_model = best_models[best_name]

# 1. Model weights
torch.save(best_model.state_dict(), f'{ARTIFACTS_DIR}/model.pth')

# 2. Tokenizer (vocab)
tokenizer.save(f'{ARTIFACTS_DIR}/tokenizer.json')

# 3. Config — what hyperparameters were used
config = {
    'model_type': best_name,
    'vocab_size': tokenizer.vocab_size,
    'embed_dim': 32,
    'hidden_dim': 64,
    'max_len': MAX_LEN,
    'pad_idx': PAD_IDX,
    'num_classes': 2,
}
with open(f'{ARTIFACTS_DIR}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Saved to {ARTIFACTS_DIR}/")
for fn in os.listdir(ARTIFACTS_DIR):
    size = os.path.getsize(f'{ARTIFACTS_DIR}/{fn}')
    print(f"  {fn:>20} ({size:,} bytes)")

## Step 8: Inference — Load Model & Predict

This is what your "production" code would do — load saved artifacts and predict on new text.

In [ ]:
class SentimentClassifier:
    """Loads model + tokenizer from disk and makes predictions."""
    
    LABEL_NAMES = {0: 'negative', 1: 'positive'}
    MODEL_REGISTRY = {
        'BoW': BoWModel,
        'EmbedMean': EmbedMeanModel,
        'EmbedMLP': EmbedMLPModel,
    }
    
    def __init__(self, artifacts_dir):
        # Load config
        with open(f'{artifacts_dir}/config.json') as f:
            self.config = json.load(f)
        
        # Load tokenizer
        self.tokenizer = Tokenizer.load(f'{artifacts_dir}/tokenizer.json')
        
        # Build model based on config and load weights
        ModelClass = self.MODEL_REGISTRY[self.config['model_type']]
        if self.config['model_type'] == 'BoW':
            self.model = ModelClass(self.config['vocab_size'])
        elif self.config['model_type'] == 'EmbedMean':
            self.model = ModelClass(
                self.config['vocab_size'],
                embed_dim=self.config['embed_dim'],
                pad_idx=self.config['pad_idx'],
            )
        else:
            self.model = ModelClass(
                self.config['vocab_size'],
                embed_dim=self.config['embed_dim'],
                hidden_dim=self.config['hidden_dim'],
                pad_idx=self.config['pad_idx'],
            )
        self.model.load_state_dict(torch.load(f'{artifacts_dir}/model.pth'))
        self.model.eval()
    
    def predict(self, text):
        """Returns (label, confidence) for one piece of text."""
        ids = self.tokenizer.encode(text, max_len=self.config['max_len'])
        x = torch.tensor([ids], dtype=torch.long)
        with torch.no_grad():
            logits = self.model(x)
            probs = F.softmax(logits, dim=1)
        pred = probs.argmax(dim=1).item()
        confidence = probs[0, pred].item()
        return self.LABEL_NAMES[pred], confidence
    
    def predict_batch(self, texts):
        return [self.predict(t) for t in texts]


# Use the saved artifacts
classifier = SentimentClassifier(ARTIFACTS_DIR)
print(f"Loaded {classifier.config['model_type']} model.")
print(f"Vocab size: {classifier.tokenizer.vocab_size}\n")

# Test predictions
test_texts = [
    "This was the best film I've ever seen.",
    "Awful movie. Total waste of money.",
    "I really enjoyed it, the acting was great.",
    "Boring, predictable, and poorly written.",
    "What a wonderful experience!",
    "Painful to watch from start to finish.",
]

print(f"{'Prediction':>10} | {'Confidence':>10} | Text")
print("-" * 80)
for text in test_texts:
    label, conf = classifier.predict(text)
    print(f"  {label:>8} | {conf*100:>9.1f}% | '{text}'")

## Step 9: Error Analysis — Where Does the Model Fail?

Look at WHICH test reviews the model gets wrong. This is more useful than just looking at accuracy.

In [ ]:
# Find all misclassified test examples + confusion matrix

errors = []
correct = 0
total = 0
confusion = np.zeros((2, 2), dtype=int)   # confusion[true_label, pred_label]

best_model.eval()
with torch.no_grad():
    for ids, labels in test_loader:
        logits = best_model(ids)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)
        for i in range(len(ids)):
            true = labels[i].item()
            pred = preds[i].item()
            confusion[true, pred] += 1
            total += 1
            if pred == true:
                correct += 1
            else:
                # Decode the text and save the error
                text = tokenizer.decode(ids[i].tolist())
                conf = probs[i, pred].item()
                errors.append((text, true, pred, conf))

print(f"Test accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print(f"Errors: {len(errors)} mistakes\n")

# Confusion matrix
print("Confusion matrix (rows = true, cols = predicted):")
print(f"             Pred Neg | Pred Pos")
print(f"  True Neg     {confusion[0,0]:>4}    |   {confusion[0,1]:>4}    (FP={confusion[0,1]})")
print(f"  True Pos     {confusion[1,0]:>4}    |   {confusion[1,1]:>4}    (FN={confusion[1,0]})")

# Show errors
if errors:
    print(f"\n--- All {len(errors)} mistakes ---")
    for text, true, pred, conf in errors:
        true_label = SentimentClassifier.LABEL_NAMES[true]
        pred_label = SentimentClassifier.LABEL_NAMES[pred]
        print(f"  TRUE: {true_label:>8}  PRED: {pred_label:>8} ({conf*100:.0f}% conf)")
        print(f"    '{text}'")
        print()
else:
    print("\n No errors! 100% accurate.")

## Step 10: Probe the Model's Limits

Here's where embeddings still fall short — try to break the model with tricky inputs:

In [ ]:
# Tricky inputs that test the model's understanding

tricky = [
    # Negation
    "This was not bad.",
    "Not amazing at all.",
    "Not the worst movie I've seen.",
    
    # Mixed
    "The acting was great but the plot was terrible.",
    "Started boring, ended amazing.",
    
    # Sarcasm-ish
    "Wow, what a 'masterpiece'.",
    
    # Unknown words
    "Cinematographic genius. Unparalleled craftsmanship.",
    
    # Very short
    "Bad.",
    "Good.",
    "Meh.",   # ambiguous
]

print(f"{'Prediction':>10} | {'Conf':>6} | Text")
print("-" * 80)
for text in tricky:
    label, conf = classifier.predict(text)
    print(f"  {label:>8} | {conf*100:>5.0f}% | '{text}'")

print("\nObservations:")
print("  Negation ('not bad') often fools us — model sees 'bad' and follows it.")
print("  Mixed reviews — model averages signals, may default to neutral.")
print("  Unknown words — model sees mostly <unk> tokens, falls back to bias.")
print("\nThese are the limits of mean-pooled word embeddings.")
print("Sequence models (Day 13+) and attention (Day 15+) will fix these.")

---

## Recap — What This Project Demonstrated

You built a complete ML system. Here's the checklist:

### Engineering practices
- ✓ **Clean Tokenizer class** (encapsulated tokenization + vocab + save/load)
- ✓ **PyTorch Dataset / DataLoader** (proper batching)
- ✓ **Reusable training function** (works for any model)
- ✓ **Train / val / test split** with no data leakage
- ✓ **Model comparison** against a baseline
- ✓ **Best-model checkpointing** + early stopping
- ✓ **Saved artifacts** (model + tokenizer + config)
- ✓ **Inference class** (`SentimentClassifier`) for using the model
- ✓ **Confusion matrix** + error analysis
- ✓ **Stress testing** with tricky inputs

### Concepts you used
- nn.Module, nn.Linear, nn.Embedding, nn.Dropout
- F.cross_entropy, F.softmax, F.relu
- DataLoader with shuffle + batching
- Adam optimizer + CosineAnnealingLR scheduler
- model.train() / model.eval() modes
- torch.no_grad() for evaluation
- state_dict() for saving

### Limitations of this approach

| Problem | What's missing | When we'll fix it |
|---------|---------------|-------------------|
| Order ignored | "not bad" looks like "bad not" | Day 13 (RNN), 15+ (attention) |
| No context | "bank" of river vs money | Day 19+ (transformers) |
| Unknown words | New movie titles → `<unk>` | Day 21 (BPE tokenization) |
| No reasoning | Mixed sentiment confuses it | Larger models, more data |

---

## Where This Caps Off

You've completed Phase 2 of the journey:

```
Phase 1: Foundations (Days 1-5)              ✓
Phase 2: Neural networks from scratch         ✓
   - Single neuron, multi-layer, training,
     nn.Module, text classification, embeddings
Phase 3: Sequence models (Day 13+)            ← NEXT
Phase 4: Build a transformer
Phase 5: Real-world LLM skills
...
```

**Tomorrow:** Day 13 — RNNs. Finally, models that respect the ORDER of words.

This is a big shift. From here on, every model we build cares about sequences. That's the path to real LLMs.